# Train the dish classifier on Google Colab (free GPU)

**Do these in order, one cell at a time (click the play button on each cell).**

0. Menu **Runtime > Change runtime type > T4 GPU > Save**.
1. Run cells 1 to 3 (setup).
2. Run the **data source** cells you can (A is automatic, B needs a free Kaggle login, C/D/E are optional top-ups).
3. Run **Prepare data**, check the table it prints, then **Train** (about 20 to 60 minutes).
4. The last cell downloads `model_package.zip`. Put `dish_model.tflite` and `labels.json` into your project's `backend/model` folder.

Every class needs roughly **150 or more good images**. Classes with fewer than 60 are dropped automatically (with a warning).

## 1. Setup

In [ ]:
!pip -q install icrawler kagglehub scikit-learn matplotlib pillow
import tensorflow as tf
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

## 2. Write the helper scripts (nothing to change)

In [ ]:
%%writefile prepare_data.py
"""Merge image folders from several sources into one clean train/val/test dataset.

Each source is a folder with one sub-folder per class (any nesting inside a class folder is fine):
  src_a/samosa/*.jpg   src_b/Poha/*.png   own_photos/idli/*.jpeg ...
Class names are normalised (lower case, spaces/hyphens -> underscore), mapped with aliases, and only classes
listed in --classes are kept. Broken or tiny images are dropped, duplicates (across sources too) are removed
BEFORE splitting so the same photo can never be in both train and test.
"""
import argparse
import hashlib
import json
import random
import shutil
from collections import defaultdict
from pathlib import Path

from PIL import Image, ImageOps

EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


def norm(name: str) -> str:
    return name.strip().lower().replace("-", "_").replace(" ", "_")


def ahash(img: Image.Image) -> int:
    g = ImageOps.grayscale(img).resize((8, 8))
    px = list(g.getdata())
    avg = sum(px) / 64
    return sum(1 << i for i, p in enumerate(px) if p > avg)


def collect(sources, classes, aliases, min_side):
    keep, stats = defaultdict(list), defaultdict(lambda: defaultdict(lambda: defaultdict(int)))
    seen_md5, seen_ah = set(), defaultdict(list)
    for src in sources:
        src = Path(src)
        sname = src.name
        for cdir in sorted(p for p in src.iterdir() if p.is_dir()):
            cls = aliases.get(norm(cdir.name), norm(cdir.name))
            if cls not in classes:
                continue
            for f in sorted(cdir.rglob("*")):
                if f.suffix.lower() not in EXT or not f.is_file():
                    continue
                try:
                    data = f.read_bytes()
                    md5 = hashlib.md5(data).hexdigest()
                    if md5 in seen_md5:
                        stats[cls][sname]["duplicate"] += 1
                        continue
                    img = Image.open(f)
                    img.verify()
                    img = Image.open(f)
                    img = ImageOps.exif_transpose(img).convert("RGB")
                except Exception:
                    stats[cls][sname]["broken"] += 1
                    continue
                if min(img.size) < min_side:
                    stats[cls][sname]["too_small"] += 1
                    continue
                ah = ahash(img)
                if any((ah ^ other).bit_count() <= 3 for other in seen_ah[cls]):  # near-duplicate (resized/re-saved copy)
                    stats[cls][sname]["duplicate"] += 1
                    continue
                seen_md5.add(md5); seen_ah[cls].append(ah)
                keep[cls].append((img, sname))
                stats[cls][sname]["kept"] += 1
    return keep, stats


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--sources", nargs="+", required=True)
    ap.add_argument("--classes", required=True, help="comma-separated dish keys, or a path to dishes.json")
    ap.add_argument("--aliases", default=None, help="JSON file {source_class_name: dish_key}")
    ap.add_argument("--out", default="data")
    ap.add_argument("--min_side", type=int, default=100)
    ap.add_argument("--min_per_class", type=int, default=60, help="classes with fewer images are dropped with a warning")
    ap.add_argument("--max_per_class", type=int, default=600)
    ap.add_argument("--max_side", type=int, default=512)
    ap.add_argument("--seed", type=int, default=42)
    a = ap.parse_args()

    if a.classes.endswith(".json"):
        classes = set(json.loads(Path(a.classes).read_text()).keys())
    else:
        classes = {norm(c) for c in a.classes.split(",")}
    aliases = {norm(k): v for k, v in json.loads(Path(a.aliases).read_text()).items()} if a.aliases else {}

    keep, stats = collect(a.sources, classes, aliases, a.min_side)
    out = Path(a.out)
    if out.exists():
        shutil.rmtree(out)
    rng = random.Random(a.seed)
    summary = {"classes": {}, "dropped_classes": {}, "sources": [Path(s).name for s in a.sources], "per_source_stats": {c: {s: dict(v) for s, v in d.items()} for c, d in stats.items()}}
    for cls in sorted(classes):
        items = keep.get(cls, [])
        if len(items) < a.min_per_class:
            summary["dropped_classes"][cls] = len(items)
            print(f"WARNING: '{cls}' has only {len(items)} usable images (< {a.min_per_class}); class dropped. Add more photos for it.")
            continue
        rng.shuffle(items)
        items = [(img, sname, i) for i, (img, sname) in enumerate(items[: a.max_per_class])]
        n = len(items); n_val = max(1, round(n * 0.15)); n_test = max(1, round(n * 0.15))
        split = {"test": items[:n_test], "val": items[n_test:n_test + n_val], "train": items[n_test + n_val:]}
        for sp, lst in split.items():
            d = out / sp / cls
            d.mkdir(parents=True, exist_ok=True)
            for img, sname, i in lst:
                img.thumbnail((a.max_side, a.max_side))
                img.save(d / f"{sname}_{i:04d}.jpg", quality=90)
        summary["classes"][cls] = {"total": n, "train": len(split["train"]), "val": len(split["val"]), "test": len(split["test"])}
    tot = {k: sum(v[k] for v in summary["classes"].values()) for k in ("total", "train", "val", "test")}
    summary["totals"] = tot
    (out / "data_summary.json").write_text(json.dumps(summary, indent=2))
    print(f"\n{'class':<18}{'total':>7}{'train':>7}{'val':>6}{'test':>6}")
    for c, v in summary["classes"].items():
        print(f"{c:<18}{v['total']:>7}{v['train']:>7}{v['val']:>6}{v['test']:>6}")
    print(f"{'TOTAL':<18}{tot['total']:>7}{tot['train']:>7}{tot['val']:>6}{tot['test']:>6}   classes kept: {len(summary['classes'])}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile train.py
"""Train the Indian-dish classifier (MobileNetV2 transfer learning) and export everything the app and the PPT need.

Dataset layout (folder names must match the dish keys in backend/app/data/dishes.json):
  data/train/<dish>/*.jpg    data/val/<dish>/*.jpg    data/test/<dish>/*.jpg

Run:  python train.py --data data --epochs 15 --fine_tune_epochs 5
Outputs: backend/model/dish_model.tflite + labels.json, and reports/ (metrics.json, accuracy_loss.png, confusion_matrix.png)
"""
import argparse
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

ROOT = Path(__file__).resolve().parent


def load(path, img, batch, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        path, label_mode="categorical", image_size=(img, img), batch_size=batch, shuffle=shuffle, seed=42)


def build(num_classes, img, weights):
    base = tf.keras.applications.MobileNetV2(input_shape=(img, img, 3), include_top=False, weights=None if weights == "none" else weights, pooling="avg")
    base.trainable = False
    inp = tf.keras.Input((img, img, 3))
    x = tf.keras.layers.Rescaling(1 / 127.5, offset=-1)(inp)  # the model rescales itself: feed raw 0-255 pixels
    x = base(x, training=False)
    x = tf.keras.layers.Dropout(0.2)(x)
    out = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    infer = tf.keras.Model(inp, out, name="dish_classifier")
    aug = tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal"), tf.keras.layers.RandomRotation(0.1),
        tf.keras.layers.RandomZoom(0.1), tf.keras.layers.RandomBrightness(0.1, value_range=(0, 255)),
    ])
    return base, infer, tf.keras.Sequential([aug, infer])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data")
    ap.add_argument("--img", type=int, default=224)
    ap.add_argument("--batch", type=int, default=32)
    ap.add_argument("--epochs", type=int, default=15)
    ap.add_argument("--fine_tune_epochs", type=int, default=5)
    ap.add_argument("--weights", default="imagenet", help="imagenet or none (none only for quick tests)")
    ap.add_argument("--out", default=str(ROOT / "backend" / "model"))
    ap.add_argument("--reports", default=str(ROOT / "ml" / "reports"))
    a = ap.parse_args()
    out, rep = Path(a.out), Path(a.reports)
    out.mkdir(parents=True, exist_ok=True); rep.mkdir(parents=True, exist_ok=True)

    d = Path(a.data)
    train, val, test = load(d / "train", a.img, a.batch, True), load(d / "val", a.img, a.batch, False), load(d / "test", a.img, a.batch, False)
    classes = train.class_names
    dj = ROOT / "backend/app/data/dishes.json"
    if dj.exists():
        missing = [c for c in classes if c not in json.loads(dj.read_text())]
        if missing:
            print(f"WARNING: no recipe in dishes.json for classes: {missing}. Add them before using the app.")
    base, infer, model = build(len(classes), a.img, a.weights)

    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
    counts = np.bincount([classes.index(Path(p).parent.name) for p in train.file_paths], minlength=len(classes))
    cw = {i: float(counts.sum() / (len(classes) * max(c, 1))) for i, c in enumerate(counts)}  # rarer class -> larger weight
    h = model.fit(train, validation_data=val, epochs=a.epochs, class_weight=cw)
    hist = {k: list(map(float, v)) for k, v in h.history.items()}
    if a.fine_tune_epochs > 0:
        base.trainable = True
        for layer in base.layers[:-30]:
            layer.trainable = False
        model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="categorical_crossentropy", metrics=["accuracy"])
        h2 = model.fit(train, validation_data=val, epochs=a.fine_tune_epochs, class_weight=cw)
        for k, v in h2.history.items():
            hist[k] += list(map(float, v))

    y_true = np.concatenate([np.argmax(y, 1) for _, y in test])
    y_pred = np.argmax(infer.predict(test), 1)
    r = classification_report(y_true, y_pred, target_names=classes, output_dict=True, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    metrics = {
        "n_train": int(sum(1 for _ in train.file_paths)), "n_val": int(len(val.file_paths)), "n_test": int(len(y_true)),
        "classes": classes, "image_size": a.img, "epochs": a.epochs, "fine_tune_epochs": a.fine_tune_epochs,
        "accuracy": float(r["accuracy"]), "precision_macro": float(r["macro avg"]["precision"]),
        "recall_macro": float(r["macro avg"]["recall"]), "f1_macro": float(r["macro avg"]["f1-score"]),
        "per_class": {c: r[c] for c in classes}, "confusion_matrix": cm.tolist(), "history": hist,
    }
    (rep / "metrics.json").write_text(json.dumps(metrics, indent=2))

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(hist["accuracy"], label="Training"); ax[0].plot(hist["val_accuracy"], label="Validation"); ax[0].set_title("Accuracy"); ax[0].set_xlabel("Epoch"); ax[0].legend()
    ax[1].plot(hist["loss"], label="Training"); ax[1].plot(hist["val_loss"], label="Validation"); ax[1].set_title("Loss"); ax[1].set_xlabel("Epoch"); ax[1].legend()
    fig.tight_layout(); fig.savefig(rep / "accuracy_loss.png", dpi=160); plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(cm, cmap="Blues"); ax.set_xticks(range(len(classes)), classes, rotation=45, ha="right"); ax.set_yticks(range(len(classes)), classes)
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Confusion matrix (test set)")
    fig.tight_layout(); fig.savefig(rep / "confusion_matrix.png", dpi=160); plt.close(fig)

    infer.save(out / "dish_model.keras")
    tfl = tf.lite.TFLiteConverter.from_keras_model(infer)
    tfl.optimizations = [tf.lite.Optimize.DEFAULT]
    (out / "dish_model.tflite").write_bytes(tfl.convert())
    (out / "labels.json").write_text(json.dumps(classes))
    print(f"\nTest accuracy {metrics['accuracy']:.3f}  macro F1 {metrics['f1_macro']:.3f}\nSaved model to {out} and reports to {rep}")


if __name__ == "__main__":
    main()

## 3. Which dishes the app knows
These names must match the recipes inside the app. Do not rename them.

In [ ]:
CLASSES = ["poha", "idli", "dosa", "samosa", "biryani", "rajma_chawal", "aloo_paratha", "dal_tadka", "chapati", "naan", "paneer_butter_masala", "palak_paneer", "chana_masala", "dal_makhani", "aloo_tikki", "butter_chicken", "gulab_jamun", "jalebi"]
print(len(CLASSES), 'dishes:', CLASSES)
import os; os.makedirs('/content/src', exist_ok=True)

## Data source A: Food-101 (automatic, public, ETH Zurich)
Food-101 has only a few Indian dishes. We take just **samosa** (the download is about 5 GB, it takes a few minutes).

In [ ]:
!wget -q --show-progress https://data.vision.ee.ethz.ch/cvl/food-101.tar.gz -O /content/food-101.tar.gz
!mkdir -p /content/src/food101
!cd /content && tar -xzf food-101.tar.gz food-101/images/samosa && mv food-101/images/samosa /content/src/food101/ && rm -rf food-101 food-101.tar.gz
!ls /content/src/food101/samosa | wc -l

## Data source B: Kaggle "Indian Food Images Dataset" (80 Indian dishes)
Needs a **free Kaggle account**. If the cell asks you to log in: kaggle.com > your profile > Settings > *Create New Token*, then upload the `kaggle.json` file when asked below.
Check which of our dishes it actually contains (the cell prints the overlap).

In [ ]:
import kagglehub, os, glob
try:
    path = kagglehub.dataset_download('iamsouravbanerjee/indian-food-images-dataset')
except Exception as e:
    print('Login needed:', e)
    from google.colab import files
    files.upload()                      # choose kaggle.json
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    path = kagglehub.dataset_download('iamsouravbanerjee/indian-food-images-dataset')
# find the folder that holds one sub-folder per dish
root = max((d for d, sub, _ in os.walk(path) if len(sub) > 20), key=lambda d: len(os.listdir(d)), default=None)
print('Dataset folder:', root)
if root:
    if not os.path.exists('/content/src/kaggle_indian'): os.symlink(root, '/content/src/kaggle_indian')
    have = sorted(set(os.listdir(root)))
    norm = lambda s: s.strip().lower().replace('-', '_').replace(' ', '_')
    print('Overlap with our dishes:', sorted(set(map(norm, have)) & set(CLASSES)))
    print('Our dishes NOT in this dataset:', sorted(set(CLASSES) - set(map(norm, have))))

## Data source C (optional): another Indian food dataset, e.g. Khana
The paper "Khana: A Comprehensive Indian Cuisine Dataset" (arXiv:2509.06006) describes one. **Check where it is hosted, whether you may use it, and its licence.** Download it to your laptop as a zip with one folder per dish, then run this cell and upload the zip. Skip this cell if you have no such zip.

In [ ]:
from google.colab import files
up = files.upload()                     # choose the .zip
for name in up:
    !mkdir -p /content/src/extra_dataset && unzip -q -o "$name" -d /content/src/extra_dataset
!ls /content/src/extra_dataset | head

## Data source D (best for idli, dosa, rajma chawal, aloo paratha): your own photos
Zip a folder like `own_photos/idli/*.jpg`, `own_photos/dosa/*.jpg` (folder names exactly as in the CLASSES list above). Photos from friends, family and your own kitchen are the most useful because they look like what real users upload. Run and upload the zip.

In [ ]:
from google.colab import files
up = files.upload()                     # choose own_photos.zip
for name in up:
    !mkdir -p /content/src/own_photos && unzip -q -o "$name" -d /content/src/own_photos
!ls /content/src/own_photos | head -30

## Data source E (optional): web image search top-up
Fills dishes that still have too few images. **Read this first:** web images can be copyrighted, wrongly labelled or duplicates. Use them for study only, do not redistribute them, and always **check the preview grid** below before training. Set the numbers for the dishes you need, or leave `TOPUP` empty to skip.

In [ ]:
TOPUP = {}   # example: {'idli': 150, 'dosa': 150, 'rajma_chawal': 150, 'aloo_paratha': 150}
from icrawler.builtin import BingImageCrawler
for cls, n in TOPUP.items():
    BingImageCrawler(storage={'root_dir': f'/content/src/web/{cls}'}, log_level=40).crawl(
        keyword=cls.replace('_', ' ') + ' indian food dish', max_num=n)
    print(cls, 'downloaded')

## 4. Prepare the data: merge all sources, remove broken and duplicate photos, split 70/15/15

In [ ]:
import json, os
ALIASES = {  # other names used by datasets -> our dish names (edit if you like)
    'idly': 'idli', 'idlis': 'idli', 'dosai': 'dosa', 'masala_dosa': 'dosa', 'plain_dosa': 'dosa',
    'paratha': 'aloo_paratha', 'aloo_parantha': 'aloo_paratha', 'rajma_rice': 'rajma_chawal',
    'chole': 'chana_masala', 'chickpea_curry': 'chana_masala', 'daal_tadka': 'dal_tadka', 'daal_makhani': 'dal_makhani',
    'paneer_makhani': 'paneer_butter_masala', 'roti': 'chapati', 'phulka': 'chapati', 'chicken_biryani': 'biryani',
}
json.dump(ALIASES, open('aliases.json', 'w'))
sources = [os.path.join('/content/src', d) for d in sorted(os.listdir('/content/src'))]
print('Sources found:', sources)
!python prepare_data.py --sources {' '.join(sources)} --classes {','.join(CLASSES)} --aliases aliases.json --out /content/data --min_per_class 60 --max_per_class 600

### Look at your data (important)
Wrong labels ruin a model. Scroll through these samples. If a class looks wrong, delete that source's folder under `/content/src`, and run **Prepare data** again.

In [ ]:
import matplotlib.pyplot as plt, random, glob
classes_kept = sorted(os.listdir('/content/data/train'))
fig, axes = plt.subplots(len(classes_kept), 6, figsize=(12, 2 * len(classes_kept)))
for r, c in enumerate(classes_kept):
    files_ = random.sample(glob.glob(f'/content/data/train/{c}/*.jpg'), 6)
    for k, f in enumerate(files_):
        axes[r][k].imshow(plt.imread(f)); axes[r][k].axis('off')
    axes[r][0].set_title(c, fontsize=9, loc='left')
plt.tight_layout(); plt.show()

## 5. Train (MobileNetV2 transfer learning, then fine-tuning)

In [ ]:
!python train.py --data /content/data --epochs 15 --fine_tune_epochs 5 --out /content/out/model --reports /content/out/reports

## 6. Results (use these numbers and pictures in your PPT slides 9 to 12)

In [ ]:
import json, shutil
from IPython.display import Image, display
m = json.load(open('/content/out/reports/metrics.json'))
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in m.items() if k in ('n_train','n_val','n_test','accuracy','precision_macro','recall_macro','f1_macro')})
display(Image('/content/out/reports/accuracy_loss.png')); display(Image('/content/out/reports/confusion_matrix.png'))
shutil.copy('/content/data/data_summary.json', '/content/out/reports/data_summary.json')

## 7. Download everything
Unzip it on your laptop. Put `model/dish_model.tflite` and `model/labels.json` into your project's `backend/model` folder, then upload them to GitHub (see the deployment guide).

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/model_package', 'zip', '/content/out')
files.download('/content/model_package.zip')